In [ ]:
from rsTMS_pipeline.data_loading.params import *
from rsTMS_pipeline.data_loading.loading_utils import *
from rsTMS_pipeline.preproc.preproc_utils import *
from rsTMS_pipeline.targeting.targeting_utils import *
from rsTMS_pipeline.plotting.plotting_utils import *

In [ ]:
time_series_lengths = [300, 600, 900, 1200]
for subj in subjects: 
    for ses in sessions: 
        print('Subject:', subj, '- Session:', ses)
        FUNC_PATH, MASK_PATH, confounds_file, ANAT_PATH, GM_PATH = load_fmriprepdata(FMRIPREP_PATH, subj, ses, space)
        bold_files = sort_by_run(FUNC_PATH)
        mask_files = sort_by_run(MASK_PATH)
        print("Sorted BOLD:", bold_files)
        print("Sorted MASK:", mask_files)  

        for i in range(len(bold_files)):
            print('Run:', i)
            for n_vols in time_series_lengths:
                print('N_vols:', n_vols)
                clean_func = index_img(clean_func, slice(0, n_vols))
                clean_func, mean_func, sample_mask, confounds = clean_bold(FUNC_PATH[i], tr=1.02)
                nscans = clean_func.shape[-1]
                sgc_masker,sgc_mask,sgc_mask_noncl = sgc_masking(FUNC_PATH[i],clean_func,nscans,radius_mm=10,seeds_sgc = [(6,16,-10)],tr=1.02)
                correlation_img, correlation_map = sgc_coorelation_map(MASK_PATH[i], clean_func, sgc_mask,)
                roi_data, roi_img=dlpfc_masking(clean_func,MASK_PATH[i],seeds_dlpfc = [(-36,39,43), (-44,40,29), (-41,16,54)])
                print('Over the DLPFC ROI')
                min_voxel_idx, min_z_value, min_mni_coord = min_target_roi(correlation_img, roi_img)
                print('Over the DLPFC ROIa and GM')
                final_projected_img = min_target_gm(correlation_img, roi_img,MASK_PATH[i])